# Lab 1 at home · Two tables of your own in DuckDB

**Due:** Sunday, October 25 at 11:59 PM · **Submit:** your repo URL on CourseWorks

In class you loaded one table (penguins), found a problem and fixed it. Now do the same with **two related tables you choose**. In Session 2 we'll join them, so pick two that can be connected.

You will:
1. Pick two related public datasets and upload them
2. Load both into DuckDB and check what DuckDB decided about each column
3. Find the column the two tables share (the **key**) and check how well they match
4. Write five SQL queries that help explain your data
5. Write observations and a dataset card for each table

## 1. Pick two related tables

**Related** means they share a column you could match rows on: an ID, a name, a ZIP code, a country code, a date.

**Example pairs**
- **World Bank:** any indicator download (e.g. GDP per country per year) comes with a second file of country details (region, income group). The shared column is the country code.
- **Our World in Data:** two charts for the same countries, e.g. life expectancy and health spending. Shared columns: country and year.
- **NYC Open Data:** 311 complaints and population by ZIP code, or the street tree census and neighborhood boundaries. Shared column: ZIP code or neighborhood.
- **Your own idea:** any two public tables that share a column.

**Rules**
- **Public, open data only.** Your repo is public, and later this data goes to a free AI tier. Nothing from work, nothing personal, nothing behind a login.
- **CSV format.** If a file is Excel, open it and save it as CSV first.
- **Each file under 100 MB.** Between 1,000 and 1,000,000 rows is a good size for the main table; the second can be small.
- **Reuse allowed.** Check each license or terms page, and note it in your dataset card.

**Places to look:** [NYC Open Data](https://opendata.cityofnewyork.us) · [data.gov](https://data.gov) · [World Bank Open Data](https://data.worldbank.org) · [Our World in Data](https://ourworldindata.org) · [Kaggle Datasets](https://www.kaggle.com/datasets) (check each license)

## 2. Upload them

1. Download both CSVs to your computer.
2. In this Codespace, drag both files into the **`data`** folder in the file list on the left.
3. Rename each to something short with no spaces, e.g. `gdp.csv` and `countries.csv` (right-click → **Rename**).
4. Fill in the cell below and run it. Table names should be short, lowercase, no spaces: you'll type them in your SQL.

In [37]:
FILE_A  = "311_sep2026.csv"     # <- your main table's file name
TABLE_A = "complaints"                   # <- a short name for it, e.g. "gdp"

FILE_B  = "population_zcta2024.csv"   # <- the related table's file name
TABLE_B = "population"                 # <- a short name for it, e.g. "countries"

In [38]:
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_A = ROOT / "data" / FILE_A
DATA_B = ROOT / "data" / FILE_B

missing = [p.name for p in (DATA_A, DATA_B) if not p.exists()]
if missing:
    print("Can't find:", ", ".join(missing))
    print("Files in data/ right now:")
    for f in sorted((ROOT / "data").iterdir()):
        print("  ", f.name)
else:
    con = duckdb.connect(str(ROOT / "lab1_home.duckdb"))
    for p in (DATA_A, DATA_B):
        print(f"Found data/{p.name} ({p.stat().st_size / 1_000_000:.1f} MB)")
    print("Connected.")

Found data/311_sep2026.csv (37.0 MB)
Found data/population_zcta2024.csv (0.0 MB)
Connected.


## 3. First look

Preview the first 10 rows of each raw file before loading.

In [39]:
con.sql(f"SELECT * FROM read_csv('{DATA_A}') LIMIT 10")

┌────────────┬─────────────────────┬─────────────────────┬─────────┬─────────────────────────┬──────────────┬───────────┬─────────────┐
│ unique_key │    created_date     │     closed_date     │ agency  │     complaint_type      │ incident_zip │  borough  │   status    │
│   int64    │      timestamp      │      timestamp      │ varchar │         varchar         │    int64     │  varchar  │   varchar   │
├────────────┼─────────────────────┼─────────────────────┼─────────┼─────────────────────────┼──────────────┼───────────┼─────────────┤
│   70250274 │ 2026-09-01 00:45:08 │ 2026-09-08 16:50:00 │ DEP     │ Water Maintenance       │        10465 │ BRONX     │ In Progress │
│   70250298 │ 2026-09-01 00:06:18 │ 2026-09-03 12:45:00 │ DEP     │ Sewer Maintenance       │        10025 │ MANHATTAN │ In Progress │
│   70250323 │ 2026-09-01 00:03:12 │ 2026-09-01 23:23:00 │ DEP     │ Noise                   │        11209 │ BROOKLYN  │ Closed      │
│   70250325 │ 2026-09-01 00:36:12 │ 2026-09-02 

In [40]:
con.sql(f"SELECT * FROM read_csv('{DATA_B}') LIMIT 10")

┌──────────┬────────────┬────────────────┬─────────────────┐
│ zip_code │ population │ population_moe │ estimate_period │
│  int64   │   int64    │     int64      │     varchar     │
├──────────┼────────────┼────────────────┼─────────────────┤
│    10001 │      30511 │           1797 │ 2020-2024       │
│    10002 │      76873 │           3494 │ 2020-2024       │
│    10003 │      53560 │           2107 │ 2020-2024       │
│    10004 │       3711 │            909 │ 2020-2024       │
│    10005 │       9439 │           1107 │ 2020-2024       │
│    10006 │       4096 │            796 │ 2020-2024       │
│    10007 │       7406 │           1151 │ 2020-2024       │
│    10009 │      60209 │           3423 │ 2020-2024       │
│    10010 │      30702 │           2567 │ 2020-2024       │
│    10011 │      48277 │           2621 │ 2020-2024       │
└──────────┴────────────┴────────────────┴─────────────────┘
  10 rows                                        4 columns

## 4. Load both and check the types

Load each file into a table, then `DESCRIBE` it. Compare each column's type with what it *should* be. A number or date column stored as `VARCHAR` (text) is the same kind of problem you fixed in class.

In [41]:
for table, path in ((TABLE_A, DATA_A), (TABLE_B, DATA_B)):
    con.sql(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv('{path}')")
    rows = con.sql(f"SELECT count(*) FROM {table}").fetchone()[0]
    print(f"{table}: {rows:,} rows loaded")

complaints: 323,630 rows loaded
population: 218 rows loaded


In [42]:
con.sql(f"DESCRIBE {TABLE_A}")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ unique_key     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ created_date   │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ closed_date    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ agency         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ complaint_type │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ incident_zip   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ borough        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ status         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [43]:
con.sql(f"DESCRIBE {TABLE_B}")

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ zip_code        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ population      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ population_moe  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ estimate_period │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

`SUMMARIZE` profiles every column at once: min, max, how many distinct values, and `null_percentage` (how much is missing).

In [44]:
con.sql(f"SUMMARIZE {TABLE_A}")

┌────────────────┬─────────────┬─────────────────────┬─────────────────────────┬───────────────┬────────────────────────────┬───────────────────┬────────────────────────────┬────────────────────────────┬────────────────────────────┬────────┬─────────────────┐
│  column_name   │ column_type │         min         │           max           │ approx_unique │            avg             │        std        │            q25             │            q50             │            q75             │ count  │ null_percentage │
│    varchar     │   varchar   │       varchar       │         varchar         │     int64     │          varchar           │      varchar      │          varchar           │          varchar           │          varchar           │ int64  │  decimal(9,2)   │
├────────────────┼─────────────┼─────────────────────┼─────────────────────────┼───────────────┼────────────────────────────┼───────────────────┼────────────────────────────┼────────────────────────────┼─────────────────

In [45]:
con.sql(f"SUMMARIZE {TABLE_B}")

┌─────────────────┬─────────────┬───────────┬───────────┬───────────────┬────────────────────┬────────────────────┬─────────┬─────────┬─────────┬───────┬─────────────────┐
│   column_name   │ column_type │    min    │    max    │ approx_unique │        avg         │        std         │   q25   │   q50   │   q75   │ count │ null_percentage │
│     varchar     │   varchar   │  varchar  │  varchar  │     int64     │      varchar       │      varchar       │ varchar │ varchar │ varchar │ int64 │  decimal(9,2)   │
├─────────────────┼─────────────┼───────────┼───────────┼───────────────┼────────────────────┼────────────────────┼─────────┼─────────┼─────────┼───────┼─────────────────┤
│ zip_code        │ BIGINT      │ 10001     │ 19034     │           241 │ 10836.788990825688 │ 850.4092218872433  │ 10168   │ 11022   │ 11360   │   218 │            0.00 │
│ population      │ BIGINT      │ 0         │ 105716    │           174 │ 39834.34403669725  │ 28835.52316659491  │ 16006   │ 37867   │ 6159

## 5. Fix what loaded wrong (if anything)

If every type looks right, write "No fixes needed" in the text cell after the next one and skip ahead.

Otherwise, open the CSV (double-click it in the `data` folder) and find **how missing values are written** in the columns that came in as text. Edit the `nullstr` list below to match, and run it for the table that needs fixing.

In [46]:
# Change TABLE_A / DATA_A to TABLE_B / DATA_B if it's the second table that needs fixing
con.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_A} AS
    SELECT * FROM read_csv('{DATA_A}', nullstr = ['NA', 'N/A', ''])
""")
con.sql(f"DESCRIBE {TABLE_A}")

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ unique_key     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ created_date   │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ closed_date    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ agency         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ complaint_type │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ incident_zip   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ borough        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ status         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [47]:
con.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_A} AS
    SELECT * FROM read_csv(
        '{DATA_A}',
        types = {{'incident_zip': 'VARCHAR'}}
    )
""")

con.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_B} AS
    SELECT * FROM read_csv(
        '{DATA_B}',
        types = {{'zip_code': 'VARCHAR'}}
    )
""")

con.sql(f"DESCRIBE {TABLE_A}").show()
con.sql(f"DESCRIBE {TABLE_B}").show()

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ unique_key     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ created_date   │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ closed_date    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ agency         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ complaint_type │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ incident_zip   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ borough        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ status         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌─────────────────┬─────────────┬─────────┬───────

**What did you fix, and how did you know?** (Double-click to edit.)
DuckDB initially inferred incident_zip and zip_code as BIGINT. I reloaded both CSV files with these columns set to VARCHAR because ZIP/ZCTA codes are identifiers rather than quantities. This preserves leading zeros. I checked both tables with DESCRIBE to confirm the change.



## 6. Find the key: how do the two tables connect?

The **key** is the column both tables share. Type its name in each table below. It may be spelled differently in each, e.g. `Country Code` in one and `iso3` in the other.

The next cell counts the distinct key values in each table and how many of them appear in both. (It treats both as text, so a ZIP code stored as a number still matches one stored as text.)

In [48]:
KEY_A = "incident_zip"   # <- the shared column's name in your main table
KEY_B = "zip_code"   # <- its name in the second table

In [49]:
con.sql(f"""
    WITH a AS (SELECT DISTINCT CAST("{KEY_A}" AS VARCHAR) AS k FROM {TABLE_A}),
         b AS (SELECT DISTINCT CAST("{KEY_B}" AS VARCHAR) AS k FROM {TABLE_B})
    SELECT
        (SELECT count(*) FROM a)                           AS keys_in_a,
        (SELECT count(*) FROM b)                           AS keys_in_b,
        (SELECT count(*) FROM a WHERE k IN (SELECT k FROM b)) AS keys_in_both
""")

┌───────────┬───────────┬──────────────┐
│ keys_in_a │ keys_in_b │ keys_in_both │
│   int64   │   int64   │    int64     │
├───────────┼───────────┼──────────────┤
│       243 │       218 │          218 │
└───────────┴───────────┴──────────────┘

**Describe the relationship** (double-click to edit):

- **Key column:** in the main table it's called complaints, uses incident_zip, in the second population, uses zip_code
- **How they relate:** does one row in the second table match *one* row in the main table, or *many*? (e.g. one country has many GDP rows, one per year)
One row in the population table can match many rows in the complaints table because multiple complaints can occur in the same ZIP/ZCTA area. This is a one-to-many relationship from population to complaints.
- **How well they match:** how many keys appear in only one table? Give one example and a guess at why (spelling, codes vs. names, regions mixed in with countries...)
There are 218 matching codes. The complaints table has 24 non-null ZIP codes that do not appear in the population table, while the population table has no codes missing from the complaints table. The reported count of 243 includes 242 non-null ZIP codes plus one NULL category. For example, 10000 appears only in complaints and may be an invalid or placeholder code. Other unmatched codes may reflect differences between postal ZIP codes and Census ZCTAs.



## 7. Five queries that explain your data

Write one query of each kind. Under each, double-click the text cell and write one line: **what does this result tell us?** Use either table.

Tip: if a column name has spaces or capitals, wrap it in double quotes: `"Country Name"`. Start from a query in the in-class notebook and change one thing.

### Query 1 · Size and shape
*How many rows, and over what time period or area?* Useful SQL: `count(*), min(), max()`

In [50]:
con.sql("""
   SELECT COUNT(*) AS total_requests,
           MIN(created_date) AS earliest_request,
           MAX(created_date) AS latest_request
    FROM complaints
""")

┌────────────────┬─────────────────────┬─────────────────────┐
│ total_requests │  earliest_request   │   latest_request    │
│     int64      │      timestamp      │      timestamp      │
├────────────────┼─────────────────────┼─────────────────────┤
│         323630 │ 2026-09-01 00:00:05 │ 2026-09-30 23:59:58 │
└────────────────┴─────────────────────┴─────────────────────┘

**What this tells us:** The dataset contains 323,630 service requests created from September 1 to September 30, 2026, confirming that its time range matches the selected month.

### Query 2 · Filter
*Which rows meet a condition you care about?* Useful SQL: `WHERE`

In [51]:
con.sql("""
   SELECT zip_code, population, estimate_period
    FROM population
    WHERE population >= 100000
    ORDER BY population DESC
""")

┌──────────┬────────────┬─────────────────┐
│ zip_code │ population │ estimate_period │
│ varchar  │   int64    │     varchar     │
├──────────┼────────────┼─────────────────┤
│ 11368    │     105716 │ 2020-2024       │
│ 11208    │     105428 │ 2020-2024       │
│ 11385    │     101704 │ 2020-2024       │
└──────────┴────────────┴─────────────────┘

**What this tells us:** hree ZCTAs in this dataset have estimated populations of at least 100,000: 11368 (105,716), 11208 (105,428), and 11385 (101,704), based on the 2020–2024 ACS five-year estimates.

### Query 3 · Group
*How does a count or average differ across categories?* Useful SQL: `GROUP BY`

In [52]:
con.sql("""
    SELECT borough,
           COUNT(*) AS request_count
    FROM complaints
    GROUP BY borough
    ORDER BY request_count DESC
""")

┌───────────────┬───────────────┐
│    borough    │ request_count │
│    varchar    │     int64     │
├───────────────┼───────────────┤
│ BROOKLYN      │        105705 │
│ QUEENS        │         80444 │
│ MANHATTAN     │         64994 │
│ BRONX         │         59213 │
│ STATEN ISLAND │         12877 │
│ Unspecified   │           397 │
└───────────────┴───────────────┘

**What this tells us:** Brooklyn has the most service requests in September 2026, with 105,705 records, followed by Queens with 80,444. Another 397 records have an unspecified borough.

### Query 4 · Rank
*What are the top or bottom 10 of something?* Useful SQL: `ORDER BY ... LIMIT 10`

In [53]:
con.sql("""
   SELECT incident_zip,
           COUNT(*) AS request_count
    FROM complaints
    WHERE incident_zip IS NOT NULL
    GROUP BY incident_zip
    ORDER BY request_count DESC, incident_zip
    LIMIT 10
""")

┌──────────────┬───────────────┐
│ incident_zip │ request_count │
│   varchar    │     int64     │
├──────────────┼───────────────┤
│ 11226        │          5616 │
│ 10457        │          4337 │
│ 11385        │          4206 │
│ 11201        │          4135 │
│ 10467        │          4087 │
│ 10458        │          3890 │
│ 10452        │          3889 │
│ 10456        │          3883 │
│ 10453        │          3849 │
│ 11207        │          3628 │
└──────────────┴───────────────┘
  10 rows            2 columns

**What this tells us:** ZIP code 11226 has the most service requests in September 2026, with 5,616 records, followed by 10457 with 4,337. This ranking compares total requests without adjusting for population.

### Query 5 · Quality check
*What's missing, duplicated or impossible?* Useful SQL: `count(*) - count(column), count(DISTINCT ...)`

In [54]:
con.sql("""
    SELECT
        COUNT(*) - COUNT(incident_zip) AS missing_zip,
        COUNT(*) - COUNT(closed_date) AS missing_closed_date,
        COUNT(unique_key) - COUNT(DISTINCT unique_key) AS duplicate_ids,
        SUM(
            CASE WHEN closed_date < created_date
                 THEN 1 ELSE 0 END
        ) AS closed_before_created
    FROM complaints
""")

┌─────────────┬─────────────────────┬───────────────┬───────────────────────┐
│ missing_zip │ missing_closed_date │ duplicate_ids │ closed_before_created │
│    int64    │        int64        │     int64     │        int128         │
├─────────────┼─────────────────────┼───────────────┼───────────────────────┤
│        2973 │               25424 │             0 │                   213 │
└─────────────┴─────────────────────┴───────────────┴───────────────────────┘

**What this tells us:** The dataset has 2,973 missing ZIP codes, 25,424 missing closure dates, no duplicate request IDs, and 213 records with closure dates earlier than creation dates. These missing and inconsistent values need investigation before geographic or response-time analysis.

## 8. Three observations

Specific things you noticed that you'd want to check before trusting this data. Point to numbers.

1. There are 2,973 requests without ZIP codes, approximately 0.92% of all records. Before comparing areas, I would check whether missing ZIP codes are concentrated in particular boroughs or complaint types.
2. There are 25,424 missing closure dates and 213 records where closure occurred before creation. Before calculating response times, I would investigate these records and decide how to handle missing or inconsistent dates.
3. Of the 242 distinct non-null complaint ZIP codes, 24 do not match the population table. Before calculating requests per 1,000 residents, I would investigate these unmatched codes and note that the 2026 requests are being compared with 2020–2024 population estimates.

**One question you'd want a non-technical person to be able to ask of this data:**
Which ZIP/ZCTA areas had the most 311 service requests per 1,000 residents in September 2026, using the available population estimates?



## 9. Dataset cards

| | Main table | Second table |
|---|---|---|
| **Dataset name** |NYC 311 Service Requests — September 2026 subset |ACS 2024 Five-Year Total Population by ZCTA — selected areas |
| **Source (link)** |https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9 |https://data.census.gov/table/ACSDT5Y2024.B01003 |
| **Who collected it, and why** |NYC311 and responding city agencies record non-emergency service requests and their handling. Published by NYC Office of Technology and Innovation (OTI). |The U.S. Census Bureau collects ACS survey data to estimate population and community characteristics. Census Reporter provides access to these Census estimates. |
| **License or terms of use** |Publicly available for reuse under the NYC Open Data Terms of Use. |Underlying Census Bureau data are generally in the U.S. public domain; see the Census Bureau policy. |
| **Rows × columns** |323,630 × 8 in the downloaded subset |218 × 4; includes only ZCTAs matching ZIP codes present in the selected 311 records |
| **Time period covered** |Requests created September 1–30, 2026; downloaded October 24, 2026. Closure dates and statuses may reflect later updates. |2020–2024 ACS five-year estimates, released January 29, 2026; downloaded October 24, 2026. These are not 2026 population estimates. |
| **How often it's updated** |The source dataset updates daily; this CSV is a fixed snapshot. |A new ACS five-year estimate release is published annually; this CSV uses the fixed 2024 edition. |

## 10. Submit

1. Run the cell below to close the connection.
2. Fill in the lifecycle map of your data in `labs/lab1.md`.
3. **Save this notebook (Cmd+S)**, then commit and push: **Source Control** → message `Lab 1` → **Commit** → **Sync Changes**.
4. Check on GitHub that both CSVs, both notebooks and `labs/lab1.md` are there, with results visible.
5. Paste your repo URL into the Lab 1 assignment on CourseWorks.

In [56]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
